Aqui vamos testar codigos para depois criar a pipeline 

In [6]:
# imports
import json
import requests
from dotenv import load_dotenv
import os

In [7]:
# carregando a chave da api
load_dotenv()

API_KEY = os.getenv('OPENWEATHER_API_KEY')
if not API_KEY:
    raise ValueError('A chave da api "OPENWEATHER_API_KEY" não foi encontrada. Verifique o arquivo')

In [8]:
city = 'São Paulo'

def get_previsao_atual(city, api_key):
    # Extrair dados metereologicos da cidade selecionada
    
    base_url = 'https://api.openweathermap.org/data/2.5/weather'
    params = {
    'q': city,
    'appid':api_key,
    'units': 'metric',
    'lang': 'pt_br'
    }
    try:
        response = requests.get(base_url, params= params)
        response.raise_for_status()
        data = response.json()
        dados_prev_atual = {}
        
        cidade_atual = data['name']
        temperatura_atual = data['main']['temp']
        temp_max = data['main']['temp_max']
        temp_min = data['main']['temp_min']
        sensacao_termica = data['main']['feels_like']

        dados_prev_atual = {
            'Cidade': cidade_atual,
            'Temperatura Atual': temperatura_atual,
            'Minima': temp_min,
            'Maxima': temp_max,
            'Sensação Termica': sensacao_termica
        }

        return dados_prev_atual
    except requests.exceptions.RequestException as e:
        print(f'Erro ao obter o clima atual para {city}:{e}')
        return None
    

In [75]:
get_previsao_atual(city,API_KEY)

{'Cidade': 'São Paulo',
 'Temperatura Atual': 13.87,
 'Minima': 13.2,
 'Maxima': 14.49,
 'Sensação Termica': 13.58}

In [20]:
def buscar_previsao_futura(city,api_key):
    base_url = 'https://api.openweathermap.org/data/2.5/forecast'
    params = params = {
    'q': city,
    'appid':api_key,
    'units': 'metric',
    'lang': 'pt_br'
    }
    try:
        response = requests.get(base_url, params)
        response.raise_for_status()
        return response.json().get('list', [])
    except requests.exceptions.RequestException as e:
        print(f'Erro ao buscar a previsão futura para {city}: {e}')
        return None

buscar_previsao_futura(city, API_KEY)

[{'dt': 1755302400,
  'main': {'temp': 14.31,
   'feels_like': 14.04,
   'temp_min': 14.31,
   'temp_max': 15.63,
   'pressure': 1025,
   'sea_level': 1025,
   'grnd_level': 934,
   'humidity': 86,
   'temp_kf': -1.32},
  'weather': [{'id': 803,
    'main': 'Clouds',
    'description': 'nublado',
    'icon': '04n'}],
  'clouds': {'all': 75},
  'wind': {'speed': 5.26, 'deg': 122, 'gust': 6.59},
  'visibility': 10000,
  'pop': 0,
  'sys': {'pod': 'n'},
  'dt_txt': '2025-08-16 00:00:00'},
 {'dt': 1755313200,
  'main': {'temp': 14.72,
   'feels_like': 14.41,
   'temp_min': 14.72,
   'temp_max': 15.54,
   'pressure': 1025,
   'sea_level': 1025,
   'grnd_level': 934,
   'humidity': 83,
   'temp_kf': -0.82},
  'weather': [{'id': 803,
    'main': 'Clouds',
    'description': 'nublado',
    'icon': '04n'}],
  'clouds': {'all': 82},
  'wind': {'speed': 4.83, 'deg': 116, 'gust': 6.65},
  'visibility': 10000,
  'pop': 0,
  'sys': {'pod': 'n'},
  'dt_txt': '2025-08-16 03:00:00'},
 {'dt': 1755324000

In [25]:
from datetime import date, timedelta, datetime
def transoformar_dados_previsao(lista_previsao):
    # Transforma dados e busca as previsoes para os proximos 3 dias usando a api e a bibilioteca datetime
    if not lista_previsao:
        return []
    hoje = date.today()
    limite_data = hoje + timedelta(days=4)
    previsoes_diarias = {}

    for previsao in lista_previsao:
        timestamp = datetime.strptime(previsao['dt_txt'],'%Y-%m-%d %H:%M:%S')
        data_previsao = timestamp.date()

        if hoje < data_previsao < limite_data:
            temp_atual = previsao['main']['temp']

            if data_previsao not in previsoes_diarias:
                previsoes_diarias[data_previsao] = {
                    'data': data_previsao.strftime('%Y-%m-%d'),
                    'temp_max':temp_atual,
                    'clima_representativo': previsao['weather'][0]['description']
                }
            else:
                if temp_atual > previsoes_diarias[data_previsao]['temp_max']:
                    previsoes_diarias[data_previsao]['temp_max'] = temp_atual
                    previsoes_diarias[data_previsao]['clima_representativo'] = previsao['weather'][0]['description']
                
    return list(previsoes_diarias.values())


In [24]:
lista_bruta = buscar_previsao_futura(city, API_KEY)
lista_previsao = transoformar_dados_previsao(lista_bruta)

print(lista_previsao)
#print(lista_bruta)

[{'data': '2025-08-16', 'temp_max': 21.43, 'clima_representativo': 'céu limpo'}, {'data': '2025-08-17', 'temp_max': 24.89, 'clima_representativo': 'céu limpo'}, {'data': '2025-08-18', 'temp_max': 22.3, 'clima_representativo': 'céu limpo'}]
